# Build a Neural Network, One Line at a Time
### CSCI 4220 · in-class coding · 50 minutes · companion to *UDL ch. 3 — Shallow neural networks*

Robby wore a fitness tracker for a whole day. We'll build Monday's network — **one hidden
unit at a time** — to learn his day. Then we'll give it harder data: two inputs, a year of
Robby's life, and finally a picture of his cat.

**The goal.** By minute 30 you will have typed every piece of this equation (slide 8):

$$y = \phi_0 + \phi_1\,a[\theta_{10} + \theta_{11}x] + \phi_2\,a[\theta_{20} + \theta_{21}x] + \phi_3\,a[\theta_{30} + \theta_{31}x]$$

| min | | slides | you write |
|---|---|---|---|
| 0–4 | **1** · Robby's day · a straight line tries | 2–5 | the best line |
| 4–7 | **2** · the activation function | 17 | `relu` |
| 7–14 | **3** · one hidden unit: θ₀ and θ₁ | 15–17 | three units |
| 14–21 | **4** · build Robby's day | 18–19 | four units + weights |
| 21–27 | **5** · see the network | 16–22, 52 | a slope |
| 27–32 | **6** · the network in UDL notation | 25, 54–56 | `network` |
| 32–38 | **7** · enough units → any shape | 25–27, 39 | `fit` |
| 38–44 | **8** · two inputs: sleep × coffee | 33–38, 51 | `network2d` |
| 44–50 | **9** · complex data: a year, then a cat | 26–27, 42–43, 49 | `count_params` |

**Blanks:** fill each `______` with the commented line directly above it.

```python
# coffee = relu(-7.5 + 1.0 * t)
coffee = ______
```

▶ **Run these two cells once** — imports, then Robby's data and the plotting helpers (no need to read them).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── helpers: run once, never edit ───────────────────────────────────────────────
from matplotlib import animation, patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML

INK, GRAY, FAINT = "#0b0b0b", "#52514e", "#b9b8b3"
EVENT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # one colour per hidden unit
VIOLET = LinearSegmentedColormap.from_list("violet", ["#f4f2fb", "#4a3aa7"])
CLOCK = ([6, 9, 12, 15, 18, 21, 24], ["6 am", "9 am", "noon", "3 pm", "6 pm", "9 pm", "midnight"])
SHORT = ([6, 12, 18, 24], ["6 am", "noon", "6 pm", "midnight"])
plt.rcParams.update({"figure.figsize": (8.5, 3.9), "axes.grid": True, "grid.alpha": .18,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
                     "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "contour.negative_linestyle": "solid"})
t = np.linspace(6, 24, 400)                                    # a smooth clock, 6 am to midnight
HALO = [pe.withStroke(linewidth=3, foreground="white")]

# ── data ──
def robbys_tracker():
    """Robby's energy (0-10), logged every 15 minutes from 6 am to midnight."""
    rng = np.random.default_rng(4220)
    when = [6, 7, 8, 10, 12.5, 14.5, 15.5, 17, 19, 21, 23, 24]
    level = [2, 2, 3.2, 8, 6.6, 3.4, 4.2, 6.4, 5.4, 3.6, 1.6, 1.2]
    fine = np.linspace(6, 24, 2000)
    k = np.exp(-0.5 * np.linspace(-3, 3, 41) ** 2)
    smooth = np.convolve(np.pad(np.interp(fine, when, level), 20, mode="edge"), k / k.sum(), mode="valid")
    hrs = np.arange(6, 24.01, 0.25)
    return hrs, np.interp(hrs, fine, smooth) + rng.normal(0, 0.35, hrs.size)

def robbys_year():
    """365 days: hours of sleep, cups of coffee, and how productive Robby was."""
    rng = np.random.default_rng(365)
    s, c = rng.uniform(3.5, 9.5, 365), rng.uniform(0, 5, 365)
    p = 2 + 3.2 * (1 - np.exp(-(s - 3) / 2)) + 1.6 * c - 0.45 * c ** 2 - 0.35 * c * np.maximum(0, 6.5 - s)
    return s, c, p + rng.normal(0, 0.45, 365)

def _tri(x, y, a, b, c):
    d1 = (x - b[0]) * (a[1] - b[1]) - (a[0] - b[0]) * (y - b[1])
    d2 = (x - c[0]) * (b[1] - c[1]) - (b[0] - c[0]) * (y - c[1])
    d3 = (x - a[0]) * (c[1] - a[1]) - (c[0] - a[0]) * (y - a[1])
    return ~(((d1 < 0) | (d2 < 0) | (d3 < 0)) & ((d1 > 0) | (d2 > 0) | (d3 > 0)))

def robbys_cat_photo():
    g = np.linspace(-1, 1, 72); x, y = np.meshgrid(g, g)
    head = (x / 0.62) ** 2 + ((y + 0.12) / 0.52) ** 2 <= 1
    ears = _tri(x, y, (-.58, .18), (-.18, .36), (-.52, .86)) | _tri(x, y, (.58, .18), (.18, .36), (.52, .86))
    eyes = (((x + .24) / .09) ** 2 + ((y + .02) / .13) ** 2 <= 1) | (((x - .24) / .09) ** 2 + ((y + .02) / .13) ** 2 <= 1)
    nose = _tri(x, y, (-.07, -.2), (.07, -.2), (0, -.29))
    return x, y, ((head | ears) & ~eyes & ~nose).astype(float)

# ── plots: Robby's day ──
def _dots(ax, hrs, lvl):
    ax.scatter(hrs, lvl, s=16, color=GRAY, alpha=.45, label="tracker", zorder=1)

def show(x, *curves, title=""):
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    for y, label, *c in curves:
        ax.plot(x, y, color=c[0] if c else INK, lw=2.6, label=label)
    ax.axhline(0, color=GRAY, lw=.8, alpha=.4); ax.set_title(title); ax.legend(frameon=False)
    plt.show()

def show_day(*curves, title=""):
    fig, ax = plt.subplots()
    _dots(ax, hours, energy)
    for y, label, *c in curves:
        ax.plot(t, y, color=c[0] if c else INK, lw=2.6, label=label)
    ax.set_xticks(*CLOCK); ax.set_ylim(-0.5, 10.5); ax.set_ylabel("energy")
    ax.set_title(title); ax.legend(frameon=False, fontsize=9, loc="upper right")
    plt.show()

def show_network(units, names, weights, baseline):
    """Slides 16-19: the hidden units -> weight each one -> add them up."""
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
    for u, name, w, c in zip(units, names, weights, EVENT):
        ax[0].plot(t, u, color=c, lw=2.3, label=name)
        ax[1].plot(t, w * u, color=c, lw=2.3, label=f"{w:+} × {name}")
        for a, y in ((ax[0], u), (ax[1], w * u)):
            i = int(np.argmax(np.abs(y)))
            a.annotate(name, (t[i], y[i]), xytext=(-4, 5), textcoords="offset points", ha="right",
                       fontsize=9, color=GRAY, path_effects=HALO)
    total = baseline + sum(w * u for w, u in zip(weights, units))
    _dots(ax[2], hours, energy); ax[2].plot(t, total, color=INK, lw=2.6, label="energy")
    for a, title in zip(ax, ["1 · the hidden units", "2 · weight each one", "3 · add them up (+ baseline)"]):
        a.set_xticks(*SHORT); a.set_title(title); a.axhline(0, color=GRAY, lw=.8, alpha=.4)
        a.legend(frameon=False, fontsize=8, loc="upper left")
    ax[2].legend(frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout(); plt.show()

def draw_network(names, theta, phi):
    """Slide 22: each parameter multiplies its source and adds to its target."""
    fig, ax = plt.subplots(figsize=(9, 4.4)); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ys = np.linspace(.86, .14, len(names)); xi, xh, xo = .08, .48, .9
    def node(x, y, text, colour):
        ax.add_patch(plt.Circle((x, y), .052, color=colour, zorder=3))
        ax.text(x, y, text, ha="center", va="center", color="white", fontsize=11, weight="bold", zorder=4)
    sub = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    for d, (yh, (t0, t1), p, name, c) in enumerate(zip(ys, theta, phi[1:], names, EVENT), start=1):
        ax.annotate("", (xh - .055, yh), (xi + .055, .5), arrowprops=dict(arrowstyle="-|>", color=FAINT, lw=1.4))
        ax.annotate("", (xo - .055, .5), (xh + .055, yh), arrowprops=dict(arrowstyle="-|>", color=c, lw=1 + .6 * abs(p)))
        node(xh, yh, f"h{d}".translate(sub), c)
        ax.text(xh, yh - .075, f"{name} · " + f"θ{d}0".translate(sub) + f" = {t0:g}", ha="center", va="top",
                fontsize=8.5, color=GRAY)
        f = .3; px = xh + .055 + f * (xo - xh - .11); py = yh + f * (.5 - yh) + (.05 if yh >= .5 else -.05)
        ax.text(px, py, f"φ{d}".translate(sub) + f" = {p:+g}", fontsize=9, color=GRAY, ha="center", va="center",
                path_effects=HALO)
    node(xi, .5, "x", INK); node(xo, .5, "y", INK)
    ax.text(xi, .41, "time of day", ha="center", fontsize=9, color=GRAY)
    ax.text(xo, .41, "energy\nφ₀ = " + f"{phi[0]:g}", ha="center", va="top", fontsize=9, color=GRAY)
    ax.text(.26, .93, "θ₁₁ = θ₂₁ = θ₃₁ = θ₄₁ = 1", ha="center", fontsize=9, color=GRAY)
    ax.set_title("Robby's day as a neural network (slide 22)")
    plt.show()

def show_pattern(names, theta, phi):
    """Slide 21: which units are on in each stretch of the day, and the slope there."""
    edges = [6] + sorted(-t0 / t1 for t0, t1 in theta) + [24]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.3), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.4]})
    y = phi[0] + sum(p * np.maximum(0, t0 + t1 * t) for (t0, t1), p in zip(theta, phi[1:]))  # before network() exists
    _dots(a1, hours, energy); a1.plot(t, y, color=INK, lw=2.6, zorder=3)
    for k, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        if k % 2:
            a1.axvspan(lo, hi, color="#efeee9", zorder=0); a2.axvspan(lo, hi, color="#efeee9", zorder=0)
        mid = (lo + hi) / 2
        slope = sum(p * t1 for (t0, t1), p in zip(theta, phi[1:]) if t0 + t1 * mid > 0)
        a1.text(mid, 9.7, f"{slope:+.1f}/h", ha="center", fontsize=9.5, color=GRAY, weight="bold")
    a1.set_ylim(-0.5, 10.6); a1.set_ylabel("energy"); a1.set_title("Activation pattern: which units are on (slide 21)")
    for row, ((t0, t1), name, c) in enumerate(zip(theta, names, EVENT)):
        a2.fill_between(t, row - .32, row + .32, where=(t0 + t1 * t) > 0, color=c, lw=0)
        a2.text(5.85, row, name, ha="right", va="center", fontsize=9, color=GRAY)
    a2.set_yticks([]); a2.set_ylim(-.6, len(names) - .4); a2.invert_yaxis(); a2.grid(False)
    a2.set_xticks(*CLOCK); a2.set_xlim(6, 24)
    fig.tight_layout(); plt.show()

def show_activations(theta, phi):
    """Slide 56: same network, different activation function a[.]."""
    acts = [("ReLU", relu), ("softplus = log(1 + eᶻ)", lambda z: np.log1p(np.exp(z))),
            ("sigmoid = 1 / (1 + e⁻ᶻ)", lambda z: 1 / (1 + np.exp(-z)))]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
    for a, (name, f) in zip(ax, acts):
        _dots(a, hours, energy); a.plot(t, network(t, theta, phi, a=f), color=INK, lw=2.4)
        a.set_title(f"a = {name}"); a.set_xticks(*SHORT); a.set_ylim(-0.5, 10.5)
    fig.tight_layout(); plt.show()

def _schedule(ax, every, hrs, lvl, grid):
    theta, phi = fit(hrs, lvl, every)
    miss = np.sqrt(np.mean((network(hrs, theta, phi) - lvl) ** 2))
    _dots(ax, hrs, lvl); ax.plot(grid, network(grid, theta, phi), color=INK, lw=2.3)
    ax.set_title(f"a unit every {every} h\n{len(theta)} units · misses the dots by {miss:.2f}", fontsize=10)

def show_schedules(everys):
    fig, ax = plt.subplots(1, len(everys), figsize=(4.4 * len(everys), 3.7), sharey=True, squeeze=False)
    for a, every in zip(ax[0], everys):
        _schedule(a, every, hours, energy, t); a.set_xticks(*SHORT); a.set_ylim(-0.5, 10.5)
    fig.tight_layout(); plt.show()

def animate_schedules(everys):
    fig, ax = plt.subplots(figsize=(8.5, 3.9))
    _dots(ax, hours, energy); (line,) = ax.plot([], [], color=INK, lw=2.6)
    ax.set_xticks(*CLOCK); ax.set_ylim(-0.5, 10.5); ax.set_xlim(5.6, 24.4)
    def frame(i):
        theta, phi = fit(hours, energy, everys[i])
        line.set_data(t, network(t, theta, phi))
        ax.set_title(f"a unit every {everys[i]} h  ·  {len(theta)} hidden units")
        return (line,)
    anim = animation.FuncAnimation(fig, frame, frames=len(everys), interval=900)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode="once"))

# ── plots: two inputs ──
SLEEP, COFFEE = np.meshgrid(np.linspace(3, 10, 260), np.linspace(0, 5, 260))

def _unit_line(ax, theta_d, name, colour, xlim=(3, 10), ylim=(0, 5)):
    t0, t1, t2 = theta_d
    if abs(t2) > 1e-9:
        xs = np.linspace(*xlim, 400); ys = -(t0 + t1 * xs) / t2
    else:
        ys = np.linspace(*ylim, 400); xs = np.full_like(ys, -t0 / t1)
    ok = (ys >= ylim[0]) & (ys <= ylim[1]) & (xs >= xlim[0]) & (xs <= xlim[1])
    ax.plot(xs[ok], ys[ok], color=colour, lw=2.4, path_effects=[pe.withStroke(linewidth=4.5, foreground="white")])
    i = np.flatnonzero(ok)[len(np.flatnonzero(ok)) * 3 // 4]
    ax.annotate(name, (xs[i], ys[i]), xytext=(6, -12), textcoords="offset points", fontsize=9.5, color=INK,
                weight="bold", path_effects=HALO)

def _plane_axes(ax):
    ax.set_xlim(3, 10); ax.set_ylim(0, 5); ax.grid(False)
    ax.set_xlabel("hours of sleep  x₁"); ax.set_ylabel("cups of coffee  x₂")

def show_units_2d(theta, names):
    """Slides 34-37: each hidden unit, drawn over the whole input plane."""
    fig, ax = plt.subplots(1, len(theta), figsize=(5 * len(theta), 3.9))
    for a, th, name, c in zip(ax, theta, names, EVENT):
        a.pcolormesh(SLEEP, COFFEE, relu(th[0] + th[1] * SLEEP + th[2] * COFFEE), cmap=VIOLET, shading="auto")
        _unit_line(a, th, name, c); _plane_axes(a)
        a.set_title(f"{name} = a[{th[0]:g} {th[1]:+g}·x₁ {th[2]:+g}·x₂]", fontsize=10)
    fig.tight_layout(); plt.show()

def show_2d(theta, phi, names):
    """Slide 38: the output is flat tilted pieces, joined along the unit lines."""
    y = network2d(SLEEP, COFFEE, theta, phi)
    fig, ax = plt.subplots(figsize=(8, 5))
    m = ax.pcolormesh(SLEEP, COFFEE, y, cmap=VIOLET, shading="auto")
    ax.contour(SLEEP, COFFEE, y, levels=14, colors=GRAY, linewidths=.6, alpha=.7)
    fig.colorbar(m, ax=ax, label="productivity", pad=.02)
    for th, name, c in zip(theta, names, EVENT):
        _unit_line(ax, th, name, c)
    _plane_axes(ax); ax.set_title("Productivity — the thin lines are equal-productivity contours")
    plt.show()

# ── plots: complex data ──
def random_units(D, seed, lo=(3, 0), hi=(10, 5)):
    """D hidden-unit lines, each through a random point at a random angle."""
    rng = np.random.default_rng(seed)
    p = rng.uniform(lo, hi, (D, 2)); ang = rng.uniform(0, 2 * np.pi, D)
    return [(-(np.cos(a) * px + np.sin(a) * py), np.cos(a), np.sin(a)) for (px, py), a in zip(p, ang)]

def fit2d(x1, x2, y, theta):
    X = np.column_stack([np.ones_like(x1)] + [relu(t0 + t1 * x1 + t2 * x2) for t0, t1, t2 in theta])
    return np.linalg.lstsq(X, y, rcond=None)[0]

def show_year():
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    lo, hi = np.percentile(prod, [2, 98])
    m = ax.scatter(sleep, coffee, c=prod, cmap=VIOLET, s=34, edgecolor=GRAY, lw=.4, vmin=lo, vmax=hi)
    fig.colorbar(m, ax=ax, label="productivity", pad=.02)
    _plane_axes(ax); ax.set_title("Robby's year: one dot per day"); plt.show()

def show_year_fits(our_theta):
    lo, hi = np.percentile(prod, [2, 98])
    fits = [("our 3 hand-built units", our_theta), ("30 random units", random_units(30, 30)),
            ("300 random units", random_units(300, 300))]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for a, (label, th) in zip(ax, fits):
        ph = fit2d(sleep, coffee, prod, th); Y = network2d(SLEEP, COFFEE, th, ph)
        a.pcolormesh(SLEEP, COFFEE, Y, cmap=VIOLET, shading="auto", vmin=lo, vmax=hi)
        a.contour(SLEEP, COFFEE, Y, levels=12, colors=GRAY, linewidths=.5, alpha=.6)
        miss = np.sqrt(np.mean((network2d(sleep, coffee, th, ph) - prod) ** 2))
        _plane_axes(a); a.set_title(f"{label}\nmisses the days by {miss:.2f}", fontsize=10)
    fig.tight_layout(); plt.show()

def show_cat_portrait(Ds):
    x, y, photo = robbys_cat_photo()
    fig, ax = plt.subplots(1, len(Ds) + 1, figsize=(4.3 * (len(Ds) + 1), 4.4))
    ax[0].imshow(photo, origin="lower", extent=(-1, 1, -1, 1), cmap=VIOLET); ax[0].set_title("Robby's cat (the data)")
    for a, D in zip(ax[1:], Ds):
        th = random_units(D, D + 1, lo=(-1, -1), hi=(1, 1))
        ph = fit2d(x.ravel(), y.ravel(), photo.ravel(), th)
        a.imshow(np.clip(network2d(x, y, th, ph), 0, 1), origin="lower", extent=(-1, 1, -1, 1), cmap=VIOLET)
        if D <= 10:
            for th_d in th:
                _unit_line(a, th_d, "", EVENT[0], xlim=(-1, 1), ylim=(-1, 1))
        a.set_title(f"{D} hidden units")
    for a in ax: a.set_xticks([]); a.set_yticks([]); a.grid(False)
    fig.tight_layout(); plt.show()

print("helpers ready")

---
## 1 · Robby's day, and a straight line tries  <small>0–4 min · slides 2–5</small>

▶ Load Robby's tracker — his energy (0–10) every 15 minutes, 6 a.m. to midnight — and plot it.

In [ ]:
hours, energy = robbys_tracker()
show_day(title="Robby's energy, every 15 minutes")

**Question for class:** What happened at 8 a.m.? Around 1? At 3?

The only model we own is 1D linear regression (slides 3–4) — two parameters:

$$y = \phi_0 + \phi_1 x$$

▶ Fit the best straight line through the readings.

In [ ]:
# slope, intercept = np.polyfit(hours, energy, 1)
slope, intercept = ______

show_day((intercept + slope * t, "best straight line"),
         title="The line's theory: Robby just slowly runs down all day")

**Takeaway:** the best possible line still misses the whole day. As slide 5 puts it, the 1D
regression model is *obviously limited* — it can't bend. So we'll build something that can,
out of lines.

---
## 2 · The activation function  <small>4–7 min · slide 17</small>

A neural network adds one ingredient to linear regression: an **activation function** $a[\cdot]$.
Ours is the ReLU (*rectified linear unit*):

$$a[z] = \mathrm{ReLU}[z] = \begin{cases} 0 & z < 0 \\ z & z \ge 0 \end{cases}$$

▶ Write ReLU.

In [ ]:
def relu(z):
    # return np.maximum(0, z)
    return ______

print(relu(np.array([-2, -1, 0, 1, 2])))       # expect [0 0 0 1 2]

▶ Plot it.

In [ ]:
z = np.linspace(-3, 3, 200)
show(z, (relu(z), "a[z] = ReLU[z]"), title="ReLU: flat, then a line")

**Question for class:** Where's the interesting part of this picture?

**Takeaway:** the corner at $z = 0$. Everything today is built out of corners.

---
## 3 · One hidden unit: θ₀ and θ₁  <small>7–14 min · slides 15–17</small>

A **hidden unit** is a line, then the activation function:

$$h = a[\theta_0 + \theta_1 x]$$

The line inside, $\theta_0 + \theta_1 x$, is the **pre-activation**.

The corner sits where the pre-activation crosses zero:

$$\theta_0 + \theta_1 x = 0 \quad\Longrightarrow\quad x = -\,\theta_0 / \theta_1$$

Robby's first coffee: θ₀ = −7.5, θ₁ = 1, so the corner is at −(−7.5)/1 = **7.5 → 7:30 a.m.**
Read it as *"nothing happens until 7:30, then start counting."*

▶ Make the coffee unit.

In [ ]:
# coffee = relu(-7.5 + 1.0 * t)
coffee = ______

show_day((coffee, "coffee = a[-7.5 + 1.0·t]", EVENT[0]), title="One hidden unit: nothing, then something")

**θ₁ sets how steep, and which way.**

- an **espresso** hits 3× faster: θ₁ = 3. Keeping the corner at 7:30 needs θ₀ = −θ₁ · 7.5 = **−22.5**
- **grogginess** is there when he wakes and gone by 9: a *negative* θ₁ faces backwards, a[9 − t]

▶ Make both, then plot all three.

In [ ]:
# espresso = relu(-22.5 + 3.0 * t)
espresso = ______
# groggy = relu(9.0 - 1.0 * t)
groggy = ______

show_day((coffee, "coffee   θ₀ = −7.5,  θ₁ = 1", EVENT[0]), (espresso, "espresso θ₀ = −22.5, θ₁ = 3", "#1d5aa6"),
         (groggy, "groggy   θ₀ = 9,     θ₁ = −1", "#4a3aa7"), title="θ₀ and θ₁ set where the corner is — and how steep")

**Question for class:** Where is each corner? Check with $-\theta_0/\theta_1$.

**Takeaway:** a hidden unit has two knobs. Together they set **where** the corner is and
**how steep / which way** the ramp goes.

---
## 4 · Build Robby's day, one unit at a time  <small>14–21 min · slides 18–19</small>

The output weights each hidden unit, then adds them up:

$$y = \phi_0 + \sum_{d=1}^{D} \phi_d\, h_d$$

**φ_d** is how strong event *d* is (and which way). **φ₀** is Robby's baseline.

▶ Baseline φ₀ = 2, plus the coffee with weight φ₁ = 2.4.

In [ ]:
# day = 2 + 2.4 * coffee
day = ______

print(f"energy at midnight: {day[-1]:.1f}")
show_day((day, "2 + 2.4 × coffee"), title="Step 1 · a baseline of 2, plus coffee")

**Question for class:** Is Robby okay? *(Check his energy at midnight.)*

Each cell below adds **one hidden unit.** Watch the midnight number.

▶ Add the crash at 10 a.m. — a **negative** weight pulls energy down.

In [ ]:
# crash = relu(-10 + 1.0 * t)
crash = ______
# day = 2 + 2.4 * coffee - 3.4 * crash
day = ______

print(f"energy at midnight: {day[-1]:.1f}")
show_day((day, "+ the crash"), title="Step 2 · the coffee wears off at 10")

**Question for class:** What's missing? *(There is always an afternoon coffee.)*

▶ Add the 2:30 coffee.

In [ ]:
# coffee2 = relu(-14.5 + 1.0 * t)
coffee2 = ______
# day = 2 + 2.4 * coffee - 3.4 * crash + 2.2 * coffee2
day = ______

print(f"energy at midnight: {day[-1]:.1f}")
show_day((day, "+ second coffee"), title="Step 3 · the 2:30 coffee")

▶ Add the evening wind-down.

In [ ]:
# evening = relu(-17 + 1.0 * t)
evening = ______
# day = 2 + 2.4 * coffee - 3.4 * crash + 2.2 * coffee2 - 1.9 * evening
day = ______

print(f"energy at midnight: {day[-1]:.1f}")
show_day((day, "the whole day"), title="Step 4 · the evening wind-down")

**Takeaway:** a shallow neural network with **D = 4** hidden units — built one line at a time.

| Robby's day | symbol | UDL name (slides 54–55) |
|---|---|---|
| the time of day | $x$ | input |
| "coffee at 7:30" | $\theta_{d0} + \theta_{d1}x$ | pre-activation |
| "nothing until then" | $a[\cdot]$ | activation function |
| an event — coffee, the crash | $h_d$ | hidden unit (activation) |
| **when** it happens | $\theta_{d0}$ | bias |
| **how strong**, and which way | $\phi_d$ | weight |
| baseline energy | $\phi_0$ | output bias |
| his energy | $y$ | output |

---
## 5 · See the network  <small>21–27 min · slides 16–22, 52</small>

▶ Draw it the way slides 16–19 do: the hidden units → weight each one → add them up.

In [ ]:
show_network([coffee, crash, coffee2, evening], ["coffee", "crash", "coffee 2", "evening"],
             weights=[2.4, -3.4, 2.2, -1.9], baseline=2)

▶ Draw it as a network diagram (slide 22): *"each parameter multiplies its source and adds to its target."*

In [ ]:
names = ["coffee", "crash", "coffee 2", "evening"]
theta = [(-7.5, 1.0), (-10, 1.0), (-14.5, 1.0), (-17, 1.0)]      # (θd0, θd1) for each hidden unit
phi = [2, 2.4, -3.4, 2.2, -1.9]                                  # φ0, φ1, ..., φ4

draw_network(names, theta, phi)

**Activation pattern** (slide 21): between two corners, the same units stay on — so the network
is just a straight line there, with slope

$$\frac{dy}{dx} = \sum_{d \text{ active}} \phi_d\, \theta_{d1}$$

▶ Show the activation pattern and the slope of every stretch.

In [ ]:
show_pattern(names, theta, phi)

▶ Check one by hand: from 10 a.m. to 2:30 p.m., only **coffee** and the **crash** are on.

In [ ]:
# slope = 2.4 * 1.0 + (-3.4) * 1.0
slope = ______

print(f"slope from 10 a.m. to 2:30 p.m.: {slope:+.1f} energy per hour")     # expect -1.0

**Question for class:** How many straight stretches does Robby's day have?

**Takeaway:** one corner per hidden unit (slide 20), so **D units → D + 1 regions** (slide 52).
Four units, five stretches — a *piecewise linear* function.

---
## 6 · The network in UDL notation  <small>27–32 min · slides 25, 54–56</small>

With D hidden units (slide 25):

$$h_d = a[\theta_{d0} + \theta_{d1}x], \qquad y = \phi_0 + \sum_{d=1}^{D} \phi_d h_d$$

▶ Write it once, for any D — one line per step on the slides.

In [ ]:
def network(x, theta, phi, a=relu):
    # z = [t0 + t1 * x for t0, t1 in theta]                    # pre-activations       (slide 16)
    z = ______
    # h = [a(zd) for zd in z]                                   # hidden units          (slide 17)
    h = ______
    # return phi[0] + sum(p * hd for p, hd in zip(phi[1:], h))  # weight and add up     (slides 18-19)
    return ______

▶ Check it rebuilds our hand-made day exactly.

In [ ]:
print("same as our hand-built day?", np.allclose(network(t, theta, phi), day))     # expect True

▶ Swap the activation function (slide 56) — same θ and φ, different $a[\cdot]$.

In [ ]:
show_activations(theta, phi)

**Question for class:** With a sigmoid, what happens to each coffee?

▶ Count the parameters: each unit has θ_d0, θ_d1 and φ_d, plus one φ₀.

In [ ]:
D = len(theta)
# params = 3 * D + 1
params = ______

print(f"{D} hidden units -> {params} parameters")      # expect 13

**Takeaway:** thirteen numbers describe Robby's entire day. Linear regression had two.

---
## 7 · Enough units → any shape  <small>32–38 min · slides 25–27, 39</small>

> *"With enough hidden units, a shallow neural network can describe any continuous function
> … to arbitrary precision."* — the universal approximation theorem (slide 27)

We place the corners on a schedule and let the computer pick every φ. With θ fixed, $y$ is
**linear in φ**, so fitting is least squares — exactly IC 2, with the hidden units as features:

$$\hat\phi = \arg\min_\phi \sum_i \Big( y_i - \phi_0 - \sum_d \phi_d\, h_{id} \Big)^2$$

▶ Write `fit`: one column per hidden unit, then solve for φ.

In [ ]:
def fit(hrs, lvl, every):
    theta = [(-s, 1.0) for s in np.arange(hrs[0], hrs[-1], every)]       # one unit every `every` hours
    X = np.column_stack([np.ones_like(hrs)] + [relu(t0 + t1 * hrs) for t0, t1 in theta])
    # phi = np.linalg.lstsq(X, lvl, rcond=None)[0]
    phi = ______
    return theta, phi

▶ Put a unit every 2 hours, fit, and plot.

In [ ]:
# theta_fit, phi_fit = fit(hours, energy, every=2)
theta_fit, phi_fit = ______

show_day((network(t, theta_fit, phi_fit), f"{len(theta_fit)} hidden units"), title="The computer's version of Robby's day")

▶ Compare four schedules: a unit every 6 h, 2 h, 1 h, and 15 minutes.

In [ ]:
show_schedules([6, 2, 1, 0.25])

**Question for class:** Which of these four is the best model of Robby?

**Takeaway:** the 15-minute model hits every dot — and it's wrong. Robby did not have an energy
crisis at 11:15, 11:30 *and* 11:45. It memorized the tracker's noise: **overfitting.**
Fitting (slide 39) is about more than hitting the dots.

▶ 🎬 Press play, or drag the slider through the schedules.

In [ ]:
animate_schedules([6, 4, 3, 2, 1.5, 1, 0.75, 0.5, 0.25])

---
## 8 · Two inputs: sleep × coffee  <small>38–44 min · slides 33–38, 51</small>

Robby's productivity depends on **two** inputs: hours of sleep $x_1$ and cups of coffee $x_2$.
Each hidden unit gets one more θ (slide 33):

$$h_d = a[\theta_{d0} + \theta_{d1}x_1 + \theta_{d2}x_2]$$

The corner is no longer a point — it's the **line** $\theta_{d0} + \theta_{d1}x_1 + \theta_{d2}x_2 = 0$
across the input plane.

▶ Write the two-input network. Only the pre-activation line changes.

In [ ]:
def network2d(x1, x2, theta, phi):
    # z = [t0 + t1 * x1 + t2 * x2 for t0, t1, t2 in theta]      # slide 33: two inputs
    z = ______
    h = [relu(zd) for zd in z]
    return phi[0] + sum(p * hd for p, hd in zip(phi[1:], h))

▶ Three hidden units, each drawn over the whole sleep–coffee plane (slides 34–37).

In [ ]:
names2 = ["rested", "caffeinated", "jitters"]
theta2 = [(-6, 1.0, 0.0),        # on once sleep passes 6 h
          (-1, 0.0, 1.0),        # on past the first cup of coffee
          (1.5, -0.6, 1.0)]      # on when coffee outruns sleep

show_units_2d(theta2, names2)

▶ Weight them and add them up: rested **+1.2**, caffeinated **+1.0**, jitters **−2.0**, baseline **3** (slide 38).

In [ ]:
# phi2 = [3, 1.2, 1.0, -2.0]
phi2 = ______

show_2d(theta2, phi2, names2)

**Question for class:** How many flat pieces (polygons) are there? *(Follow where the contours kink.)*

With two inputs, D lines cut the plane into at most (slide 51, *"binomial coefficients!"*):

$$\binom{D}{0} + \binom{D}{1} + \binom{D}{2} \;=\; 1 + D + \frac{D(D-1)}{2}$$

▶ Write it, and check your count.

In [ ]:
def max_regions(D):
    # return 1 + D + D * (D - 1) // 2
    return ______

print("3 units, 2 inputs:", max_regions(3), "regions")     # expect 7

**Takeaway:** same network, one more input. Corners become lines, and the output becomes flat
tilted pieces — **convex polygons** (slide 38).

---
## 9 · More complex data: a year, then a cat  <small>44–50 min · slides 26–27, 42–43, 49</small>

Robby actually logged sleep, coffee and productivity **every day for a year.**

▶ Load the year and plot it — one dot per day.

In [ ]:
sleep, coffee, prod = robbys_year()
show_year()

▶ Fit it three ways: our 3 hand-built units (the computer only re-picks φ), then 30 and 300 units.

In [ ]:
show_year_fits(theta2)

**Question for class:** Which one would you trust to plan Robby's week?

**Takeaway:** complex data needs **more** hidden units — 3 is too few. But 300 units for 365 days
memorizes the year and invents nonsense between the dots.

▶ The finale — the most complex data yet: a photo of Robby's cat. Each hidden unit is one line
across the photo; the network paints a polygon patchwork.

In [ ]:
show_cat_portrait([10, 100, 1000])

**Question for class:** With 1000 units, how many polygons *could* the cat be made of?

In general — $D_i$ inputs, $D$ hidden units, $D_o$ outputs (slide 42):

$$h_d = a\Big[\theta_{d0} + \sum_{i=1}^{D_i} \theta_{di}\, x_i\Big], \qquad y_j = \phi_{j0} + \sum_{d=1}^{D} \phi_{jd}\, h_d$$

▶ Count the parameters for any shallow network — slide 43's question included.

In [ ]:
def count_params(Di, D, Do):
    # return D * (Di + 1) + Do * (D + 1)
    return ______

print("slide 43  (3 in, 3 units, 2 out):  ", count_params(3, 3, 2))          # expect 20
print("Robby's day (1 in, 4 units, 1 out):", count_params(1, 4, 1))          # expect 13
print("the cat   (2 in, 1000 units, 1 out):", count_params(2, 1000, 1))      # expect 4001
print("cat polygons, at most:              ", max_regions(1000))             # expect 500501

**Takeaway:** 4,001 parameters, up to half a million polygons — one shallow layer, drawing a cat.

---

## What you built today

```python
def relu(z):
    return np.maximum(0, z)

def network(x, theta, phi, a=relu):
    z = [t0 + t1 * x for t0, t1 in theta]                    # pre-activations
    h = [a(zd) for zd in z]                                   # hidden units
    return phi[0] + sum(p * hd for p, hd in zip(phi[1:], h))  # weight and add up
```

- **A hidden unit** is a line, then a corner: $a[\theta_0 + \theta_1 x]$, corner at $-\theta_0/\theta_1$.
- **A network** weights and adds them: one corner per unit, $D+1$ straight pieces.
- **More inputs** turn corners into lines and pieces into polygons.
- **More units** fit more complex data — until they start memorizing it.

**Next** (slide 58): *what happens if we feed one neural network into another?* → deep networks.